# Vision-Language-Action notebook

This notebook extends the Ollama chatbot pattern with RGBD perception and a guarded action model. It detects an object on a table, asks an Ollama vision model for a structured grasp target, predicts the next `ik_ee_pose`, and only moves the arm when Execute is pressed.


In [ ]:
import os
import sys
from pathlib import Path

NOTEBOOK_DIR = Path.cwd().resolve()
MODULES_DIR = NOTEBOOK_DIR.parent
ROOT_DIR = MODULES_DIR.parent
for path in (str(MODULES_DIR), str(ROOT_DIR), str(MODULES_DIR / "scripts")):
    if path not in sys.path:
        sys.path.insert(0, path)

IFACE = os.environ.get("G1_IFACE", "eth0")
DOMAIN_ID = int(os.environ.get("G1_DOMAIN_ID", "0"))
print(f"Configured for iface={IFACE!r}, domain_id={DOMAIN_ID}.")


Import Ollama, RGBD, notebook display, and IK helpers.


In [ ]:
import base64
import json
import math
import re
import time
import urllib.error
import urllib.request

import cv2
import ipywidgets as widgets
import numpy as np
from IPython.display import display

from arm_sdk import ArmSdk
from sdk_client import Robot


Configure the local models and robot clients. The vision model must support Ollama image input.


In [ ]:
OLLAMA_URL = os.environ.get("G1_OLLAMA_URL", "http://127.0.0.1:11434").rstrip("/")
CHAT_MODEL = os.environ.get("G1_CHAT_MODEL", "qwen3.5:9b")
VISION_MODEL = os.environ.get("G1_VISION_MODEL", "qwen2.5vl:7b")
RGBD_HOST = os.environ.get("G1_RGBD_HOST", "192.168.2.41")
RGBD_PORT = int(os.environ.get("G1_RGBD_PORT", "5555"))
RGBD_TOPIC = os.environ.get("G1_RGBD_TOPIC", "")
ARM = os.environ.get("G1_VLA_ARM", "right")

robot = Robot(
    iface=IFACE,
    domain_id=DOMAIN_ID,
    safety_boot=False,
    recover_dev_mode_on_init=False,
    auto_start_sensors=False,
    rgbd_host=RGBD_HOST,
    rgbd_port=RGBD_PORT,
    rgbd_topic=RGBD_TOPIC,
)
ik = ArmSdk(iface=IFACE, domain_id=DOMAIN_ID)
ik.resync()

vla_messages = [{
    "role": "system",
    "content": "You are a concise robot manipulation assistant. Return compact, valid JSON when asked.",
}]
last_prediction = None
print(f"VLA ready: ollama={OLLAMA_URL} chat={CHAT_MODEL} vision={VISION_MODEL} rgbd=tcp://{RGBD_HOST}:{RGBD_PORT}")


Ollama helpers copied from the chatbot notebook, plus JSON extraction for model outputs.


In [ ]:
def clean_reply(text):
    text = str(text).strip()
    while "<think>" in text and "</think>" in text:
        before, rest = text.split("<think>", 1)
        _hidden, after = rest.split("</think>", 1)
        text = (before + after).strip()
    return " ".join(text.split())


def post_ollama_chat(body, timeout=45.0):
    data = json.dumps(body).encode("utf-8")
    request = urllib.request.Request(
        f"{OLLAMA_URL}/api/chat",
        data=data,
        headers={"Content-Type": "application/json"},
        method="POST",
    )
    try:
        with urllib.request.urlopen(request, timeout=float(timeout)) as response:
            return json.loads(response.read().decode("utf-8"))
    except urllib.error.HTTPError as exc:
        detail = exc.read().decode("utf-8", errors="replace")
        raise RuntimeError(f"Ollama HTTP {exc.code}: {detail}") from exc


def extract_json_object(text):
    text = str(text).strip()
    match = re.search(r"```(?:json)?\s*(\{.*?\})\s*```", text, flags=re.S)
    if match:
        text = match.group(1)
    else:
        start = text.find("{")
        end = text.rfind("}")
        if start >= 0 and end > start:
            text = text[start:end + 1]
    return json.loads(text)


Perception helpers. The classical detector finds a colored object blob and estimates depth; the vision model can refine the semantic label and target choice.


In [ ]:
def jpeg_data_url(bgr):
    ok, buf = cv2.imencode(".jpg", bgr)
    if not ok:
        return ""
    return "data:image/jpeg;base64," + base64.b64encode(buf.tobytes()).decode("ascii")


def colorize_depth(depth_m, max_depth_m=4.0):
    valid = np.isfinite(depth_m) & (depth_m > 0)
    disp = np.zeros(depth_m.shape[:2], dtype=np.uint8)
    disp[valid] = np.clip(depth_m[valid] / max_depth_m * 255.0, 0, 255).astype(np.uint8)
    return cv2.applyColorMap(disp, cv2.COLORMAP_JET)


def detect_table_object(rgb_bgr, depth_m, hsv_low, hsv_high, min_area_px=500, max_depth_m=2.0):
    hsv = cv2.cvtColor(rgb_bgr, cv2.COLOR_BGR2HSV)
    mask = cv2.inRange(hsv, np.array(hsv_low, dtype=np.uint8), np.array(hsv_high, dtype=np.uint8))
    valid_depth = np.isfinite(depth_m) & (depth_m > 0.05) & (depth_m < float(max_depth_m))
    mask = mask & valid_depth.astype(np.uint8) * 255
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, np.ones((5, 5), np.uint8))
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    overlay = rgb_bgr.copy()
    if not contours:
        return None, overlay
    contour = max(contours, key=cv2.contourArea)
    area = float(cv2.contourArea(contour))
    if area < float(min_area_px):
        return None, overlay
    x, y, w, h = cv2.boundingRect(contour)
    cv2.rectangle(overlay, (x, y), (x + w, y + h), (0, 255, 255), 2)
    roi_depth = depth_m[y:y + h, x:x + w]
    valid = roi_depth[np.isfinite(roi_depth) & (roi_depth > 0)]
    median_depth = float(np.median(valid)) if valid.size else None
    cx = int(x + w / 2)
    cy = int(y + h / 2)
    return {
        "visible": True,
        "bbox_xywh": [int(x), int(y), int(w), int(h)],
        "center_px": [int(cx), int(cy)],
        "area_px": area,
        "median_depth_m": median_depth,
        "image_size": [int(rgb_bgr.shape[1]), int(rgb_bgr.shape[0])],
    }, overlay


def rotation_matrix_to_rpy(R):
    sy = math.sqrt(float(R[0, 0]) * float(R[0, 0]) + float(R[1, 0]) * float(R[1, 0]))
    if sy > 1e-6:
        roll = math.atan2(float(R[2, 1]), float(R[2, 2]))
        pitch = math.atan2(-float(R[2, 0]), sy)
        yaw = math.atan2(float(R[1, 0]), float(R[0, 0]))
    else:
        roll = math.atan2(-float(R[1, 2]), float(R[1, 1]))
        pitch = math.atan2(-float(R[2, 0]), sy)
        yaw = 0.0
    return [roll, pitch, yaw]


def current_pose(arm=ARM):
    T = ik.get_ee_pose(arm)
    xyz = [float(v) for v in T[:3, 3]]
    rpy = rotation_matrix_to_rpy(T[:3, :3])
    return {
        "arm": arm,
        "ik_ee_pose": [round(float(v), 4) for v in xyz + rpy],
        "rotation_matrix": [[round(float(v), 4) for v in row] for row in T[:3, :3]],
    }


VLA prediction. The model sees the RGB image plus detector/depth context and must return the next target pose as JSON.


In [ ]:
VLA_SYSTEM = """You are controlling a Unitree humanoid arm in base_link coordinates.
Return only valid JSON. Do not include markdown.
Predict a cautious next ik_ee_pose to position the selected hand around the detected object on the table.
The pose is [x, y, z, roll, pitch, yaw] in meters/radians. Keep one step small and reachable.
Prefer a pre-grasp pose 8 to 12 cm from the object, slightly above its center, with a neutral wrist.
If detection is unsafe or absent, set safe_to_move false and keep the current pose."""

VLA_SCHEMA = {
    "object_detected": True,
    "object_label": "object",
    "object_center_px": [0, 0],
    "confidence": 0.0,
    "next_ik_ee_pose": [0, 0, 0, 0, 0, 0],
    "safe_to_move": False,
    "reason": "short explanation",
}


def ask_vision_action(rgb_jpeg, context):
    prompt = (
        "Detect the object on the table and predict the next ik_ee_pose for the robot hand. "
        "Return JSON matching this schema: "
        f"{json.dumps(VLA_SCHEMA)}\n"
        f"Context JSON: {json.dumps(context, ensure_ascii=False)}"
    )
    body = {
        "model": VISION_MODEL,
        "messages": [
            {"role": "system", "content": VLA_SYSTEM},
            {"role": "user", "content": prompt, "images": [base64.b64encode(rgb_jpeg).decode("ascii")]},
        ],
        "stream": False,
        "think": False,
        "options": {"temperature": 0.1, "num_predict": 220},
    }
    result = post_ollama_chat(body, timeout=60.0)
    text = clean_reply(result.get("message", {}).get("content", ""))
    return extract_json_object(text), text


def sanitize_prediction(prediction, fallback_pose):
    out = dict(prediction)
    pose = out.get("next_ik_ee_pose")
    if not isinstance(pose, list) or len(pose) != 6:
        out["next_ik_ee_pose"] = list(fallback_pose)
        out["safe_to_move"] = False
        out["reason"] = "Model did not return a 6-value next_ik_ee_pose."
        return out
    pose = [float(v) for v in pose]
    cur = np.array(fallback_pose, dtype=np.float64)
    nxt = np.array(pose, dtype=np.float64)
    delta = nxt - cur
    delta[:3] = np.clip(delta[:3], -0.06, 0.06)
    delta[3:] = np.clip(delta[3:], -0.20, 0.20)
    out["next_ik_ee_pose"] = [round(float(v), 4) for v in (cur + delta)]
    out["safe_to_move"] = bool(out.get("safe_to_move")) and bool(out.get("object_detected"))
    return out


def pose_increment_from_prediction(prediction, current_xyzrpy):
    target = np.array(prediction["next_ik_ee_pose"], dtype=np.float64)
    current = np.array(current_xyzrpy, dtype=np.float64)
    return target - current


Run the panel. Tune HSV for the table object, then Detect and Predict. Execute sends a single guarded IK increment.


In [ ]:
h_low = widgets.IntSlider(value=0, min=0, max=179, description="H low")
h_high = widgets.IntSlider(value=12, min=0, max=179, description="H high")
s_low = widgets.IntSlider(value=80, min=0, max=255, description="S low")
s_high = widgets.IntSlider(value=255, min=0, max=255, description="S high")
v_low = widgets.IntSlider(value=50, min=0, max=255, description="V low")
v_high = widgets.IntSlider(value=255, min=0, max=255, description="V high")
min_area = widgets.IntSlider(value=500, min=50, max=10000, step=50, description="Area")
max_depth = widgets.FloatSlider(value=2.0, min=0.2, max=6.0, step=0.1, description="Depth m")
detect = widgets.Button(description="Detect and Predict", button_style="success")
execute = widgets.Button(description="Execute IK Step", button_style="warning")
resync = widgets.Button(description="Resync IK")
rgb_img = widgets.HTML(value="")
depth_img = widgets.HTML(value="")
status = widgets.Textarea(layout=widgets.Layout(width="100%", height="300px"), disabled=True)


def set_status(payload):
    status.value = json.dumps(payload, indent=2, default=str)


def on_detect(_=None):
    global last_prediction
    try:
        frame = robot.get_rgbd(timeout=2.0)
        rgb_bgr = frame["rgb_bgr"]
        depth_m = frame["depth_m"]
        detection, overlay = detect_table_object(
            rgb_bgr,
            depth_m,
            (h_low.value, s_low.value, v_low.value),
            (h_high.value, s_high.value, v_high.value),
            min_area.value,
            max_depth.value,
        )
        rgb_img.value = f'<img src="{jpeg_data_url(overlay)}" style="max-width:100%;"/>'
        depth_img.value = f'<img src="{jpeg_data_url(colorize_depth(depth_m, max_depth.value))}" style="max-width:100%;"/>'
        pose = current_pose(ARM)
        current_xyzrpy = pose["ik_ee_pose"]
        context = {
            "detector": detection or {"visible": False},
            "rgbd": {
                "source": frame["source"],
                "center_depth_m": frame["center_depth_m"],
                "valid_depth_fraction": frame["valid_depth_fraction"],
                "near_coverage_1m": frame["near_coverage_1m"],
            },
            "current_pose": {"arm": ARM, "ik_ee_pose": current_xyzrpy},
        }
        if not detection:
            last_prediction = {"safe_to_move": False, "reason": "No object matched the detector.", "context": context}
            set_status(last_prediction)
            return
        prediction, raw_text = ask_vision_action(frame["rgb_jpeg"], context)
        prediction = sanitize_prediction(prediction, current_xyzrpy)
        last_prediction = {"prediction": prediction, "context": context, "raw_model_text": raw_text}
        set_status(last_prediction)
    except Exception as exc:
        last_prediction = None
        set_status({"error": str(exc)})


def on_execute(_):
    try:
        if not last_prediction or "prediction" not in last_prediction:
            raise RuntimeError("Run Detect and Predict first.")
        prediction = last_prediction["prediction"]
        if not prediction.get("safe_to_move"):
            raise RuntimeError(f"Prediction is not safe_to_move: {prediction.get('reason')}")
        current_xyzrpy = last_prediction["context"]["current_pose"]["ik_ee_pose"]
        inc = pose_increment_from_prediction(prediction, current_xyzrpy)
        info = ik.ik_move_EE(inc, arm=ARM, position_only=False, max_dq=0.12)
        set_status({"executed_increment": [round(float(v), 4) for v in inc], "ik_result": info, "prediction": prediction})
    except Exception as exc:
        set_status({"error": str(exc), "last_prediction": last_prediction})


def on_resync(_):
    ik.resync()
    set_status({"status": "IK resynced", "pose": current_pose(ARM)})

detect.on_click(on_detect)
execute.on_click(on_execute)
resync.on_click(on_resync)
set_status({"status": "ready", "arm": ARM})
display(widgets.VBox([
    widgets.HBox([h_low, h_high, s_low, s_high, v_low, v_high]),
    widgets.HBox([min_area, max_depth, detect, execute, resync]),
    status,
    widgets.HBox([rgb_img, depth_img]),
]))
